In [ ]:
import pandas as pd
import os
from datawrapper import Datawrapper
dw = Datawrapper(access_token=os.environ['DATAWRAPPER_KEY'])


In [4]:
df = pd.read_csv('data/vv_capture_2025-12-08_16-50-43/cleaned_verifier_data.csv', converters={'FIPS code': str}).convert_dtypes()
pollbooks = ['Paper Poll Book', 'In-House Electronic Poll Book','Commercial Electronic Poll Book']
epbs = df[df["Equipment Type"].isin(pollbooks)]
display(epbs.sample(5))

/var/folders/2f/3k2b0y_s79n84y37bcnb5t5h0000gn/T/ipykernel_5032/2945375718.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/vv_capture_2025-12-08_16-50-43/cleaned_verifier_data.csv', converters={'FIPS code': str}).convert_dtypes()


,Year,FIPS code,State,Jurisdiction,Registered Voters,Equipment Type,Manufacturer,Model,First Year in Use,Years in Use,...,Election Day Accessible,Early Voting Standard,Early Voting Accessible,Mail Ballot/Absentee Equipment,Notes on usage,Precincts,Voting Location,All Mail Ballot?,Election Day Marking Method,Election Day Tabulation
183970,2012,4712900000,Tennessee,Morgan County,11852,Paper Poll Book,Not Applicable,Not Applicable,<NA>,<NA>,...,False,True,False,False,<NA>,10,Assigned Polling Place,False,DREs without VVPAT for all voters,DRE
15436,2022,0900365370,Connecticut,Town of Rocky Hill (Hartford County),13313,Paper Poll Book,Not Applicable,Not Applicable,<NA>,<NA>,...,False,False,False,False,<NA>,3,Assigned Polling Place,False,Hand marked paper ballots and BMDs,Optical Scan
25804,2006,1311700000,Georgia,Forsyth County,85138,Commercial Electronic Poll Book,Diebold,ExpressPoll,<NA>,<NA>,...,False,True,False,False,<NA>,30,Assigned Polling Place,False,DREs without VVPAT for all voters,DRE
40910,2006,1814900000,Indiana,Starke County,17562,Paper Poll Book,Not Applicable,Not Applicable,<NA>,<NA>,...,False,True,False,False,<NA>,21,Assigned Polling Place,False,DREs without VVPAT for all voters,DRE
49021,2014,2005500000,Kansas,Finney County,15274,Paper Poll Book,Not Applicable,Not Applicable,<NA>,<NA>,...,False,True,False,False,<NA>,29,Assigned Polling Place,False,Hand marked paper ballots and BMDs,Optical Scan


In [ ]:
# check to see how many jurisdiction/year combinations have double counted voters (because they use an EPB and a paper poll book)
# looking at E-Day reduces but does not eliminate the duplicate voter problem.
# Up to 11% of voters are double counted depending on the year...


In [5]:
# one solution: choose an ordering and drop duplicates accordingly
df = epbs.copy()
# Define hierarchy order
priority = {
    "Commercial Electronic Poll Book": 3,
    "In-House Electronic Poll Book": 2,
    "Paper Poll Book": 1
}

# Map to a priority column
df["pb_priority"] = df["Equipment Type"].map(priority)

# For each jurisdiction, keep only its highest-priority pollbook type
df_primary = (
    df.sort_values("pb_priority", ascending=False)
      .drop_duplicates(subset=["FIPS code", "Year"], keep="first")
)

# Final cleaned column
df_primary["Pollbook_Class"] = df_primary["Equipment Type"]


In [6]:
# Count unique Equipment Types per jurisdiction per year
equipment_counts = epbs.groupby(["Year", "FIPS code"])["Equipment Type"].nunique().reset_index()

# Filter to those with more than one type
multiple_types = equipment_counts[equipment_counts["Equipment Type"] > 1]

# Filter the main dataframe to only those jurisdictions with multiple equipment types
multi_fips = multiple_types[["Year", "FIPS code"]]
df_multi = epbs.merge(multi_fips, on=["Year", "FIPS code"], how="inner")

# Sum registered voters for these jurisdictions
voters_multi = df_multi.groupby("Year")["Registered Voters"].sum().reset_index()
voters_multi.rename(columns={"Registered Voters": "Voters in Multi-Type Jurisdictions"}, inplace=True)

# For context, also compute total registered voters per year
total_voters = epbs.drop_duplicates(['FIPS code', 'Year']).groupby(['Year'])['Registered Voters'].sum()

# Merge and compute the percentage
summary = voters_multi.merge(total_voters, on="Year")
summary["Pct Affected"] = summary["Voters in Multi-Type Jurisdictions"] / summary["Registered Voters"] * 100

print(summary)


    Year  Voters in Multi-Type Jurisdictions  Registered Voters  Pct Affected
0   2006                              213180          180602026      0.118039
1   2008                             1421524          179330691      0.792683
2   2010                            19836056          176188570     11.258424
3   2012                            13227393          179622618      7.363991
4   2014                            10100953          176645027      5.718221
5   2016                            14528677          193817697      7.496053
6   2018                            15478855          194259788      7.968121
7   2020                            13460242          210075856      6.407325
8   2022                            11058857          206641977      5.351699
9   2024                            12195168          211732032      5.759718
10  2026                            13524180          211738681      6.387203


In [8]:
epbs.groupby(['Year', 'Equipment Type'])['Registered Voters'].sum().unstack()

Equipment Type,Commercial Electronic Poll Book,In-House Electronic Poll Book,Paper Poll Book
Year,,,
2006,15171060,9232763,156198203
2008,16844542,13664168,148821981
2010,28590121,24590074,123008375
2012,39895013,24306866,115420739
2014,50934734,26530449,99179844
2016,72027093,28202545,93588059
2018,88097752,31342568,74819468
2020,134410762,30344567,45320527
2022,142171754,30745647,33724576


In [9]:
grouped = epbs.groupby(["Year", "Equipment Type"])["Registered Voters"].sum().rename("VotersByType")
merged = grouped.reset_index().merge(total_voters, on="Year")
merged["Pct"] = merged["VotersByType"] / merged["Registered Voters"]
result = merged.pivot(index="Year", columns="Equipment Type", values="Pct").fillna(0)


In [11]:
result = result * 100

print(result)

Equipment Type  Commercial Electronic Poll Book  \
Year                                              
2006                                   8.400271   
2008                                   9.393006   
2010                                  16.227001   
2012                                  22.210462   
2014                                  28.834513   
2016                                  37.162289   
2018                                  45.350483   
2020                                  63.982013   
2022                                  68.801004   
2024                                  70.217424   
2026                                  71.155061   

Equipment Type  In-House Electronic Poll Book  Paper Poll Book  
Year                                                            
2006                                 5.112215        86.487514  
2008                                 7.619537        82.987458  
2010                                13.956679         69.81632  
2012       

In [37]:
df = result[result.columns[:2]].reset_index()
df.columns.name=None
df

,Year,Commercial Electronic Poll Book,In-House Electronic Poll Book
0,2006,8.400271,5.112215
1,2008,9.393006,7.619537
2,2010,16.227001,13.956679
3,2012,22.210462,13.532186
4,2014,28.834513,15.019075
5,2016,37.162289,14.551068
6,2018,45.350483,16.134357
7,2020,63.982013,14.444576
8,2022,68.801004,14.878703
9,2024,70.217424,15.022338


In [ ]:
id = '86rqN'
print(dw.get_chart(id)['publicUrl'])
dw.update_chart(id, data=df)
